# Contributing a New Unlearning Technique to eval-unlearn

This notebook is a step-by-step guide for contributors who want to add a new
concept-unlearning technique to the **eval-unlearn** library.

It covers:
1. The required file structure
2. How to write a `Config` dataclass
3. How to write a `Wrapper` class with the `generate()` interface
4. How to register the technique as a plugin via `pyproject.toml`
5. **Validation tests** — the full suite your technique must pass before a PR is merged

Each test cell is self-contained and prints a clear pass/fail verdict. Run all
cells top-to-bottom to verify your implementation.

> **GPU recommended** for the smoke-test cells, but all structural/unit tests
> run on CPU with mocked pipelines.

## Prerequisites

```bash
# Install eval-unlearn in editable mode so your new package is importable
pip install -e /path/to/eval-unlearn-testing/Packages/eval-unlearn
pip install pytest
```

Your technique's external package (if any) must also be installed and importable.

In [ ]:
import os

os.environ["HF_HOME"] = "/vol/bitbucket/m24/.cache"
os.environ["HF_DATASETS_CACHE"] = os.path.join(os.environ["HF_HOME"], "datasets")
os.environ["MPLCONFIGDIR"] = "/vol/bitbucket/m24/.cache/matplotlib"

# ── Set this to your new technique's registered name ────────────────────────
TECHNIQUE_NAME = "my_technique"   # e.g. "my_technique"

# ── Module path (dotted) to your wrapper class ───────────────────────────────
TECHNIQUE_MODULE = f"eval_unlearn.techniques.{TECHNIQUE_NAME}.wrapper"  # adjust if different
TECHNIQUE_CLASS  = "MyTechnique"   # the class name inside wrapper.py

# ── External package that your wrapper imports ───────────────────────────────
EXTERNAL_PACKAGE = "my_technique_pkg"   # the pip-installable package name
EXTERNAL_CLASS   = "MyPipeline"         # the pipeline class inside that package

# ── Minimal valid config kwargs ───────────────────────────────────────────────
# Must be enough to instantiate the Config with no errors.
REQUIRED_CONFIG = {
    "erase_concept": "nudity",
    "device": "cpu",
}

print(f"Technique name  : {TECHNIQUE_NAME}")
print(f"Module path     : {TECHNIQUE_MODULE}")
print(f"External package: {EXTERNAL_PACKAGE}")

---
## Part 1 — File Structure

Every technique lives in its own sub-package inside `src/eval_unlearn/techniques/`.
Create the following files:

```
src/eval_unlearn/techniques/
└── my_technique/
    ├── __init__.py      # (empty or re-export wrapper)
    ├── config.py        # Config dataclass
    └── wrapper.py       # Wrapper class with generate()
```

### `config.py` template

```python
from dataclasses import dataclass
from typing import Optional
from ...configs.base import BaseConfig

@dataclass(frozen=True)
class MyTechniqueConfig(BaseConfig):
    erase_concept: str = ""
    device: str = "cuda"
    num_inference_steps: int = 50
    guidance_scale: float = 7.5
    # Add any technique-specific hyperparameters here
    my_strength: float = 1.0

    def __post_init__(self):
        if not self.erase_concept:
            raise ValueError("erase_concept must not be empty")
        if self.my_strength <= 0:
            raise ValueError("my_strength must be > 0")

    @classmethod
    def from_dict(cls, data):
        return super().from_dict(data)
```

### `wrapper.py` template

```python
from typing import Any, List, Optional
from ...registry import register_technique
from ...logging_utils import get_logger
from .config import MyTechniqueConfig

logger = get_logger(__name__)

try:
    import my_technique_pkg as mtp
except ImportError as e:
    raise ImportError(
        "MyTechnique requires 'my_technique_pkg'. "
        "Install it from its repo and re-install eval-unlearn."
    ) from e

@register_technique("my_technique")
class MyTechnique:
    def __init__(self, **kwargs):
        self.config = MyTechniqueConfig.from_dict(kwargs)
        self.pipe   = mtp.MyPipeline(concept=self.config.erase_concept)

    def generate(
        self,
        prompts: List[str],
        seed: Optional[int] = None,
        **kwargs,
    ) -> List[Any]:
        logger.info(f"Generating {len(prompts)} images ('{self.config.erase_concept}' erased)")
        return self.pipe.generate(
            prompts=prompts,
            seed=seed,
            num_inference_steps=self.config.num_inference_steps,
            guidance_scale=self.config.guidance_scale,
        )
```

---
## Part 2 — Plugin Registration

Add two entries to `pyproject.toml`:

**1. Entry point** (so `eval-unlearn plugins` picks it up automatically):
```toml
[project.entry-points."eval_unlearn.techniques"]
my_technique = "eval_unlearn.techniques.my_technique.wrapper:MyTechnique"
```

**2. Base model** (if your technique uses a fixed diffusion backbone, add it to
`src/eval_unlearn/techniques/_base_models.py`):
```python
TECHNIQUE_BASE_MODELS = {
    ...
    "my_technique": "CompVis/stable-diffusion-v1-4",
}
```

After editing `pyproject.toml` re-install the package:
```bash
pip install -e .
```

---
## Part 3 — Validation Tests

The cells below are the **exact checks** the maintainers run against every new
technique. Each cell prints `PASS` or `FAIL` with a reason.

All tests must show `PASS` before you open a pull request.

In [ ]:
# ── Shared helpers ────────────────────────────────────────────────────────────
import sys
import importlib
import traceback
from unittest.mock import MagicMock, patch
from PIL import Image

def _dummy_image():
    return Image.new("RGB", (64, 64), color=(100, 149, 237))

def _mock_pipeline(return_images=None):
    """Return a mock external-package pipeline whose .generate() returns dummy images."""
    pipe = MagicMock()
    imgs = return_images or [_dummy_image()]
    pipe.generate.return_value = imgs
    return pipe

def _reload_wrapper():
    """Drop and reload the wrapper module so patches take effect cleanly."""
    sys.modules.pop(TECHNIQUE_MODULE, None)
    return importlib.import_module(TECHNIQUE_MODULE)

def _pass(label):
    print(f"  ✓  PASS  {label}")

def _fail(label, reason):
    print(f"  ✗  FAIL  {label}")
    print(f"           Reason: {reason}")

def _run(label, fn):
    try:
        fn()
        _pass(label)
    except AssertionError as e:
        _fail(label, str(e))
    except Exception as e:
        _fail(label, traceback.format_exc(limit=3))

print("Helpers loaded.")

### Test Group 1 — Config Dataclass

The `Config` must:
- Be importable as `eval_unlearn.techniques.<name>.config`
- Accept all required kwargs without error
- Reject invalid values with `ValueError`
- Round-trip through `from_dict` / `to_dict`

In [ ]:
print("=" * 60)
print("Test Group 1: Config Dataclass")
print("=" * 60)

CONFIG_MODULE = f"eval_unlearn.techniques.{TECHNIQUE_NAME}.config"
CONFIG_CLASS  = f"{TECHNIQUE_CLASS}Config"   # convention: wrapper class + "Config"

# --------------------------------------------------------------------------
# T1.1  Config module is importable
# --------------------------------------------------------------------------
def t1_1_config_importable():
    mod = importlib.import_module(CONFIG_MODULE)
    assert hasattr(mod, CONFIG_CLASS), (
        f"{CONFIG_CLASS} not found in {CONFIG_MODULE}. "
        f"Available names: {[x for x in dir(mod) if not x.startswith('_')]}"
    )

_run("T1.1  Config module is importable", t1_1_config_importable)

# --------------------------------------------------------------------------
# Load config class for subsequent tests
# --------------------------------------------------------------------------
try:
    _config_mod = importlib.import_module(CONFIG_MODULE)
    ConfigCls   = getattr(_config_mod, CONFIG_CLASS)
except Exception as e:
    print(f"  [SKIP remaining config tests — cannot import config: {e}]")
    ConfigCls = None

# --------------------------------------------------------------------------
# T1.2  Instantiation with required kwargs succeeds
# --------------------------------------------------------------------------
def t1_2_instantiation():
    assert ConfigCls is not None, "Config class could not be loaded (see T1.1)"
    cfg = ConfigCls(**REQUIRED_CONFIG)
    assert cfg is not None

_run("T1.2  Instantiation with required kwargs succeeds", t1_2_instantiation)

# --------------------------------------------------------------------------
# T1.3  Empty erase_concept raises ValueError
# --------------------------------------------------------------------------
def t1_3_empty_concept_raises():
    assert ConfigCls is not None
    import pytest
    bad_kwargs = {**REQUIRED_CONFIG, "erase_concept": ""}
    try:
        ConfigCls(**bad_kwargs)
        raise AssertionError(
            "Expected ValueError for empty erase_concept, but no error was raised."
        )
    except ValueError:
        pass  # correct

_run("T1.3  Empty erase_concept raises ValueError", t1_3_empty_concept_raises)

# --------------------------------------------------------------------------
# T1.4  from_dict round-trips correctly
# --------------------------------------------------------------------------
def t1_4_from_dict_roundtrip():
    assert ConfigCls is not None
    cfg1 = ConfigCls(**REQUIRED_CONFIG)
    cfg2 = ConfigCls.from_dict(REQUIRED_CONFIG)
    assert cfg1.erase_concept == cfg2.erase_concept, (
        f"from_dict mismatch: cfg1.erase_concept={cfg1.erase_concept!r}, "
        f"cfg2.erase_concept={cfg2.erase_concept!r}"
    )

_run("T1.4  from_dict round-trips correctly", t1_4_from_dict_roundtrip)

# --------------------------------------------------------------------------
# T1.5  to_dict returns a dict with erase_concept
# --------------------------------------------------------------------------
def t1_5_to_dict():
    assert ConfigCls is not None
    cfg = ConfigCls(**REQUIRED_CONFIG)
    d   = cfg.to_dict()
    assert isinstance(d, dict), f"to_dict() must return dict, got {type(d)}"
    assert "erase_concept" in d, "to_dict() result must contain 'erase_concept'"
    assert d["erase_concept"] == REQUIRED_CONFIG["erase_concept"]

_run("T1.5  to_dict returns dict with erase_concept", t1_5_to_dict)

# --------------------------------------------------------------------------
# T1.6  from_dict ignores unknown keys gracefully
# --------------------------------------------------------------------------
def t1_6_from_dict_ignores_extras():
    assert ConfigCls is not None
    extra_kwargs = {**REQUIRED_CONFIG, "__nonexistent_key__": 999}
    cfg = ConfigCls.from_dict(extra_kwargs)   # must not raise
    assert cfg is not None

_run("T1.6  from_dict ignores unknown keys", t1_6_from_dict_ignores_extras)

### Test Group 2 — Wrapper Class & `generate()` Interface

The `Wrapper` must:
- Be importable and decorated with `@register_technique`
- Expose a `generate(prompts, seed, **kwargs) -> List[PIL.Image]` method
- Pass `seed` through to the underlying pipeline
- Return a list with one image per prompt
- Raise (not silently fail) if the pipeline crashes

In [ ]:
print("=" * 60)
print("Test Group 2: Wrapper Class & generate()")
print("=" * 60)

def _make_technique(mock_imgs=None):
    """Instantiate the wrapper with the external package fully mocked."""
    mock_pipe = _mock_pipeline(mock_imgs or [_dummy_image()])
    mock_pkg  = MagicMock()
    setattr(mock_pkg, EXTERNAL_CLASS, MagicMock(return_value=mock_pipe))

    with patch.dict("sys.modules", {EXTERNAL_PACKAGE: mock_pkg}):
        mod = _reload_wrapper()
        cls = getattr(mod, TECHNIQUE_CLASS)
        tech = cls(**REQUIRED_CONFIG)
    return tech, mock_pipe

# --------------------------------------------------------------------------
# T2.1  Wrapper module is importable (with mocked external package)
# --------------------------------------------------------------------------
def t2_1_wrapper_importable():
    mock_pkg = MagicMock()
    with patch.dict("sys.modules", {EXTERNAL_PACKAGE: mock_pkg}):
        mod = _reload_wrapper()
    assert hasattr(mod, TECHNIQUE_CLASS), (
        f"{TECHNIQUE_CLASS} not found in {TECHNIQUE_MODULE}."
    )

_run("T2.1  Wrapper module is importable", t2_1_wrapper_importable)

# --------------------------------------------------------------------------
# T2.2  generate() returns a non-empty list of images
# --------------------------------------------------------------------------
def t2_2_generate_returns_images():
    tech, _ = _make_technique([_dummy_image(), _dummy_image()])
    result  = tech.generate(["prompt A", "prompt B"])
    assert isinstance(result, list), f"generate() must return list, got {type(result)}"
    assert len(result) == 2, f"Expected 2 images, got {len(result)}"

_run("T2.2  generate() returns a non-empty list of images", t2_2_generate_returns_images)

# --------------------------------------------------------------------------
# T2.3  One prompt → one image
# --------------------------------------------------------------------------
def t2_3_one_prompt_one_image():
    tech, _ = _make_technique([_dummy_image()])
    result  = tech.generate(["a single prompt"])
    assert len(result) == 1, f"Expected 1 image for 1 prompt, got {len(result)}"

_run("T2.3  One prompt → one image", t2_3_one_prompt_one_image)

# --------------------------------------------------------------------------
# T2.4  Seed is forwarded to the pipeline
# --------------------------------------------------------------------------
def t2_4_seed_forwarded():
    tech, mock_pipe = _make_technique()
    tech.generate(["prompt"], seed=42)
    mock_pipe.generate.assert_called_once()
    _, kwargs = mock_pipe.generate.call_args
    assert kwargs.get("seed") == 42 or 42 in mock_pipe.generate.call_args[0], (
        "seed=42 was not passed through to the underlying pipeline. "
        f"Pipeline was called with args={mock_pipe.generate.call_args}"
    )

_run("T2.4  seed is forwarded to the pipeline", t2_4_seed_forwarded)

# --------------------------------------------------------------------------
# T2.5  generate() raises when the external package is unavailable at import
# --------------------------------------------------------------------------
def t2_5_import_error_on_missing_package():
    with patch.dict("sys.modules", {EXTERNAL_PACKAGE: None}):
        sys.modules.pop(TECHNIQUE_MODULE, None)
        try:
            importlib.import_module(TECHNIQUE_MODULE)
            raise AssertionError(
                "Expected ImportError or RuntimeError when external package is missing, "
                "but the module imported successfully. Make sure wrapper.py raises "
                f"if '{EXTERNAL_PACKAGE}' cannot be imported."
            )
        except (ImportError, RuntimeError, ModuleNotFoundError):
            pass  # correct — the wrapper advertises its dependency clearly

_run("T2.5  ImportError raised when external package missing", t2_5_import_error_on_missing_package)

# --------------------------------------------------------------------------
# T2.6  generate() propagates pipeline exceptions
# --------------------------------------------------------------------------
def t2_6_pipeline_exception_propagates():
    tech, mock_pipe = _make_technique()
    mock_pipe.generate.side_effect = RuntimeError("pipeline crash")
    try:
        tech.generate(["bad prompt"])
        raise AssertionError(
            "Expected RuntimeError to propagate from the pipeline, but generate() "
            "returned silently. Do not swallow pipeline exceptions."
        )
    except RuntimeError:
        pass  # correct

_run("T2.6  Pipeline exceptions propagate from generate()", t2_6_pipeline_exception_propagates)

### Test Group 3 — Registry & Entry Point

The technique must be discoverable by the eval-unlearn plugin system after
`pip install -e .` (or after `importlib.metadata` picks up the entry point).

In [ ]:
print("=" * 60)
print("Test Group 3: Registry & Entry Point")
print("=" * 60)

# --------------------------------------------------------------------------
# T3.1  @register_technique decorator registers the class in the local registry
# --------------------------------------------------------------------------
def t3_1_registered_in_local_registry():
    mock_pkg = MagicMock()
    with patch.dict("sys.modules", {EXTERNAL_PACKAGE: mock_pkg}):
        _reload_wrapper()   # importing triggers @register_technique

    from eval_unlearn.registry import get_technique
    cls = get_technique(TECHNIQUE_NAME)
    assert cls is not None, (
        f"get_technique('{TECHNIQUE_NAME}') returned None. "
        "Make sure your wrapper uses @register_technique('{TECHNIQUE_NAME}')."
    )

_run("T3.1  @register_technique registers in local registry", t3_1_registered_in_local_registry)

# --------------------------------------------------------------------------
# T3.2  Entry point is listed in pyproject.toml
# --------------------------------------------------------------------------
def t3_2_entry_point_in_pyproject():
    import pathlib, tomllib
    root = pathlib.Path("../../../")
    pyproject_path = root / "pyproject.toml"

    if not pyproject_path.exists():
        # Try relative to notebook location
        pyproject_path = pathlib.Path("../../../../pyproject.toml")

    assert pyproject_path.exists(), (
        f"pyproject.toml not found at {pyproject_path.resolve()}. "
        "Adjust the path if running from a different directory."
    )

    with open(pyproject_path, "rb") as f:
        data = tomllib.load(f)

    eps = data.get("project", {}).get("entry-points", {}).get("eval_unlearn.techniques", {})
    assert TECHNIQUE_NAME in eps, (
        f"'{TECHNIQUE_NAME}' not found under [project.entry-points.\"eval_unlearn.techniques\"] "
        f"in pyproject.toml. Found: {list(eps.keys())}"
    )

_run("T3.2  Entry point declared in pyproject.toml", t3_2_entry_point_in_pyproject)

# --------------------------------------------------------------------------
# T3.3  Entry point dotted path resolves to the correct class
# --------------------------------------------------------------------------
def t3_3_entry_point_path_resolves():
    import pathlib, tomllib
    root = pathlib.Path("../../../")
    pyproject_path = root / "pyproject.toml"
    if not pyproject_path.exists():
        pyproject_path = pathlib.Path("../../../../pyproject.toml")

    with open(pyproject_path, "rb") as f:
        data = tomllib.load(f)

    eps  = data["project"]["entry-points"]["eval_unlearn.techniques"]
    spec = eps[TECHNIQUE_NAME]               # e.g. "eval_unlearn.techniques.my_technique.wrapper:MyTechnique"
    mod_path, cls_name = spec.rsplit(":", 1)

    mock_pkg = MagicMock()
    with patch.dict("sys.modules", {EXTERNAL_PACKAGE: mock_pkg}):
        mod = importlib.import_module(mod_path)

    assert hasattr(mod, cls_name), (
        f"Entry point '{spec}' specifies class '{cls_name}' but it was not found "
        f"in module '{mod_path}'. Check the class name and module path."
    )

_run("T3.3  Entry point path resolves to the wrapper class", t3_3_entry_point_path_resolves)

### Test Group 4 — BaseConfig Inheritance

The config must extend `eval_unlearn.configs.base.BaseConfig` (a frozen dataclass).
This ensures compatibility with the CLI, runners, and serialisation helpers.

In [ ]:
print("=" * 60)
print("Test Group 4: BaseConfig Inheritance")
print("=" * 60)

from eval_unlearn.configs.base import BaseConfig

# --------------------------------------------------------------------------
# T4.1  Config inherits from BaseConfig
# --------------------------------------------------------------------------
def t4_1_inherits_base_config():
    assert ConfigCls is not None, "Config class not loaded (see T1.1)"
    assert issubclass(ConfigCls, BaseConfig), (
        f"{CONFIG_CLASS} does not inherit from eval_unlearn.configs.base.BaseConfig. "
        "Change the class definition to: "
        f"class {CONFIG_CLASS}(BaseConfig):"
    )

_run("T4.1  Config class inherits from BaseConfig", t4_1_inherits_base_config)

# --------------------------------------------------------------------------
# T4.2  Config is a frozen dataclass (immutable)
# --------------------------------------------------------------------------
def t4_2_config_is_frozen():
    assert ConfigCls is not None
    cfg = ConfigCls(**REQUIRED_CONFIG)
    try:
        cfg.erase_concept = "changed"
        raise AssertionError(
            "Config must be frozen (immutable). "
            "Add frozen=True to @dataclass: @dataclass(frozen=True)"
        )
    except Exception as e:
        if "cannot assign" in str(e).lower() or "frozen" in str(e).lower() or "FrozenInstanceError" in type(e).__name__:
            pass  # correct — frozen dataclass rejects mutation
        else:
            raise

_run("T4.2  Config is a frozen (immutable) dataclass", t4_2_config_is_frozen)

# --------------------------------------------------------------------------
# T4.3  Config has 'erase_concept' and 'device' fields
# --------------------------------------------------------------------------
def t4_3_required_fields_exist():
    assert ConfigCls is not None
    import dataclasses
    field_names = {f.name for f in dataclasses.fields(ConfigCls)}
    for required in ("erase_concept", "device"):
        assert required in field_names, (
            f"Config is missing required field '{required}'. "
            f"Existing fields: {sorted(field_names)}"
        )

_run("T4.3  Config has required fields: erase_concept, device", t4_3_required_fields_exist)

### Test Group 5 — Integration with SingleBenchmarkRunner

Verifies that the technique can be driven end-to-end through the standard runner
with a lightweight metric (`clip_score`, `limit=1`).

> **Requires a GPU and a working external package install.** Skip if running
> in a CPU-only CI environment.

In [ ]:
print("=" * 60)
print("Test Group 5: SingleBenchmarkRunner Integration (mocked)")
print("=" * 60)

# --------------------------------------------------------------------------
# T5.1  Runner accepts the technique name without raising
# --------------------------------------------------------------------------
def t5_1_runner_accepts_technique():
    from eval_unlearn.runners import SingleBenchmarkRunner

    mock_pkg = MagicMock()
    with patch.dict("sys.modules", {EXTERNAL_PACKAGE: mock_pkg}):
        _reload_wrapper()   # ensure technique is registered

    # Constructing the runner must not raise
    runner = SingleBenchmarkRunner(
        technique_name   = TECHNIQUE_NAME,
        metric_name      = "clip_score",
        technique_config = REQUIRED_CONFIG,
        metric_config    = {"device": "cpu", "limit": 1},
        output_dir       = f"/tmp/eval_unlearn_test_{TECHNIQUE_NAME}",
        seed             = 0,
    )
    assert runner is not None

_run("T5.1  SingleBenchmarkRunner accepts technique name", t5_1_runner_accepts_technique)

# --------------------------------------------------------------------------
# T5.2  runner.run() produces a report dict with metric_result
# --------------------------------------------------------------------------
def t5_2_runner_run_returns_report():
    from eval_unlearn.runners import SingleBenchmarkRunner
    from eval_unlearn.types import MetricResult

    # Mock the technique's generate() so we don't need real GPU or weights
    mock_pkg = MagicMock()
    mock_pkg_cls = MagicMock()
    mock_pipe    = _mock_pipeline([_dummy_image()])
    setattr(mock_pkg, EXTERNAL_CLASS, MagicMock(return_value=mock_pipe))

    # Also mock the CLIP metric's model so no download is needed
    with patch.dict("sys.modules", {EXTERNAL_PACKAGE: mock_pkg}), \
         patch("eval_unlearn.metrics.clip_score.metric.CLIPModel") as mock_clip_cls, \
         patch("eval_unlearn.metrics.clip_score.metric.CLIPProcessor") as mock_proc_cls, \
         patch("eval_unlearn.metrics.clip_score.metric.torch") as mock_torch:

        mock_torch.cuda.is_available.return_value = False
        mock_model = MagicMock()
        mock_clip_cls.from_pretrained.return_value = mock_model
        mock_model.to.return_value = mock_model
        mock_proc_cls.from_pretrained.return_value = MagicMock()

        _reload_wrapper()

        runner = SingleBenchmarkRunner(
            technique_name   = TECHNIQUE_NAME,
            metric_name      = "clip_score",
            technique_config = REQUIRED_CONFIG,
            metric_config    = {"device": "cpu", "limit": 1},
            output_dir       = f"/tmp/eval_unlearn_test_{TECHNIQUE_NAME}",
            seed             = 0,
        )

        with patch.object(runner.technique, "generate", return_value=[_dummy_image()]), \
             patch.object(runner.metric, "update"), \
             patch.object(runner.metric, "compute",
                          return_value=MetricResult(name="CLIPScore", value=0.0)):
            report = runner.run()

    assert isinstance(report, dict), f"run() must return a dict, got {type(report)}"
    assert "metric_result" in report, (
        f"report dict missing 'metric_result' key. Keys found: {list(report.keys())}"
    )

_run("T5.2  runner.run() returns a report dict with metric_result", t5_2_runner_run_returns_report)

### Test Group 6 — `_base_models.py` Registration (if applicable)

If your technique uses a **fixed** diffusion backbone (not user-configurable),
it must be listed in `TECHNIQUE_BASE_MODELS` in
`src/eval_unlearn/techniques/_base_models.py`.

Set `REGISTERS_BASE_MODEL = True` below if this applies to your technique.

In [ ]:
# ── Set to True if your technique uses a fixed backbone model ─────────────
REGISTERS_BASE_MODEL = False  # <── change to True if applicable
EXPECTED_BASE_MODEL  = "CompVis/stable-diffusion-v1-4"  # <── adjust if different

print("=" * 60)
print("Test Group 6: _base_models.py Registration")
print("=" * 60)

# --------------------------------------------------------------------------
# T6.1  Technique is listed in TECHNIQUE_BASE_MODELS (if fixed backbone)
# --------------------------------------------------------------------------
def t6_1_base_model_registered():
    from eval_unlearn.techniques._base_models import TECHNIQUE_BASE_MODELS
    assert TECHNIQUE_NAME in TECHNIQUE_BASE_MODELS, (
        f"'{TECHNIQUE_NAME}' not found in TECHNIQUE_BASE_MODELS in "
        "src/eval_unlearn/techniques/_base_models.py. "
        f"Add: \"{TECHNIQUE_NAME}\": \"{EXPECTED_BASE_MODEL}\""
    )
    actual = TECHNIQUE_BASE_MODELS[TECHNIQUE_NAME]
    assert actual == EXPECTED_BASE_MODEL, (
        f"Base model mismatch: expected '{EXPECTED_BASE_MODEL}', got '{actual}'"
    )

if REGISTERS_BASE_MODEL:
    _run("T6.1  Technique listed in TECHNIQUE_BASE_MODELS", t6_1_base_model_registered)
else:
    print("  –  T6.1  Skipped (REGISTERS_BASE_MODEL=False)")

---
## Summary

Run the cell below to get a consolidated pass/fail view. **All groups must show
PASS** before opening a pull request.

In [ ]:
print("=" * 60)
print(f"Contribution checklist for technique: {TECHNIQUE_NAME!r}")
print("=" * 60)

checklist = [
    "src/eval_unlearn/techniques/{name}/__init__.py exists",
    "src/eval_unlearn/techniques/{name}/config.py defines {name}Config(BaseConfig)",
    "src/eval_unlearn/techniques/{name}/wrapper.py defines {cls} with @register_technique",
    "wrapper.generate(prompts, seed, **kwargs) -> List[PIL.Image]",
    "Config raises ValueError for empty erase_concept",
    "Config.from_dict / to_dict round-trips cleanly",
    "Config is frozen (@dataclass(frozen=True))",
    "Entry point added to pyproject.toml under eval_unlearn.techniques",
    "pip install -e . completed after editing pyproject.toml",
    "Optional: TECHNIQUE_BASE_MODELS updated in _base_models.py",
]

for item in checklist:
    line = item.format(name=TECHNIQUE_NAME, cls=TECHNIQUE_CLASS)
    print(f"  [ ]  {line}")

print()
print("Once all tests above show PASS and this checklist is complete,")
print("open a pull request against the main branch.")
print()
print("To run the full automated test suite:")
print("  cd /path/to/eval-unlearn && pytest -m 'not integration' -v")